# Figure 6

Paper panels with figure-specific PDF and CSV exports. Run from top to bottom. Original analysis notebooks are preserved.

## Setup

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager
from IPython.display import IFrame, display
from matplotlib.transforms import Bbox
import itertools
import os
import shutil
import pandas as pd
import KinematicPlot as kp
from group_config_new import build_groups
from survival_stats_runner import SurvivalStatsRunner

# All persistent exports belong to this paper figure, with one folder per panel.
ROOT = Path.cwd()
if not (ROOT / "KinematicPlot.py").is_file():
    raise RuntimeError("Run this notebook from the repository root.")
FIGURE_NUMBER = 6
NOTEBOOK_OUTPUT_DIR = ROOT / "Figures" / f"Figure{FIGURE_NUMBER}"
SC_DATA_DIR = ROOT / "SC data"
N_PERM = 20000
plotter = kp.PlotCreator()
stats_runner = SurvivalStatsRunner(tau=0.71, random_state=0, platform_offset=0.03, radius=0.07, fps=250)

def output_folder(*parts):
    # Create output folders only when the notebook is executed.
    folder = NOTEBOOK_OUTPUT_DIR.joinpath(*parts)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def panel_prefix(panel, name):
    return output_folder(f"Figure{panel}") / f"Figure{panel}_{name}"

def show_pdf(path, width=950, height=700):
    display(IFrame(src=Path(path).relative_to(ROOT).as_posix(), width=width, height=height))

@contextmanager
def working_directory(folder):
    # Preserve the working directory for legacy notebook loader calls.
    previous = Path.cwd()
    os.chdir(folder)
    try:
        yield
    finally:
        os.chdir(previous)

In [4]:
chr_lp_intensity_colors = {"low": "#F4A3A3", "medium": "#D73027", "high": "#7F0000"}

## AN groups and controls

In [6]:
# MTGal4 is the configuration key for the empty-Gal4 x GtACR control.
chr_an_keys = {"low": ["ANxCHR-400uW", "ADxChr-400uW"], "medium": ["ANxChr-4mW"], "high": ["ANxCHR-12mW"]}
gtacr_an_keys = ["ANxGTACR", "MTGal4"]
required_keys = [key for keys in chr_an_keys.values() for key in keys] + gtacr_an_keys
groups = build_groups(group_keys=required_keys, skip_missing=False, require_kinematics=False)
for group in groups.values():
    group.initialize_manual_data()
    group.filter_opto_data(min_trial_num=8)

## Figure 6F - AN CsChrimson and AD control landing probability

In [8]:
for intensity, keys in chr_an_keys.items():
    for key in keys:
        prefix = panel_prefix("6F", key + "_LP_ON_OFF")
        # Each group retains its paired OFF/ON test and intensity color.
        plotter.plot_LP_summary_light_from_group(groups[key], str(prefix), color=chr_lp_intensity_colors[intensity], min_trial_num=8, n_perm=N_PERM)
        show_pdf(str(prefix) + "-LP.pdf")

## Figure 6G - AN GtACR and empty-Gal4 control landing probability

In [10]:
for key in gtacr_an_keys:
    prefix = panel_prefix("6G", key + "_LP_ON_OFF")
    plotter.plot_LP_summary_light_from_group(groups[key], str(prefix), color="green", min_trial_num=8, n_perm=N_PERM)
    show_pdf(str(prefix) + "-LP.pdf")